# Two chambers and a narrow neck

A probe can fit inside both chambers yet fail to move between them. This
symmetric **dumbbell** asks when one enclosed region becomes two, and prepares
an independent reference for DFND's chamber/throat hierarchy.

We keep one fixed atomic input and use probes of **1.4, 2.2 and 6 Å**. This
notebook checks the geometry without importing or running DFND. The separate
[DFND notebook](dfnd.ipynb) executes the public API and inspects its actual
`Topography`, including native supports and copied hierarchy motifs.

Run each notebook independently from this directory in a fresh Python 3.14
kernel. Expand code inputs on the website when useful. Run All does not rewrite
retained evidence or submit coordinates to an external provider.

In [ ]:
from pathlib import Path
from html import escape
import hashlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import pyunitwizard as puw
import study

CASE = Path.cwd()
ARTIFACTS = CASE / 'artifacts/dumbbell_v1'
fixture = study.read(ARTIFACTS / 'input.json')
policy = study.read(ARTIFACTS / 'protocol.json')
assert hashlib.sha256((ARTIFACTS / 'input.json').read_bytes()).hexdigest() == policy['input_sha256']
centers = np.asarray(puw.get_value(study.quantity(fixture['coordinates'], 'coordinates'), to_unit='angstroms'))
radii = np.asarray(puw.get_value(study.quantity(fixture['atom_radii'], 'atom_radii'), to_unit='angstroms'))
anchors = study.quantity(policy['anchors'], 'anchors')
anchor_values = np.asarray(puw.get_value(anchors, to_unit='angstroms'))
assert centers.shape == (94, 3) and np.allclose(radii, 1.88, rtol=0, atol=1e-14)
assert fixture['source_atom_indices'] == list(range(94))

def table(headers, rows):
    def cells(values, tag):
        return ''.join(f'<{tag}>{escape(str(value))}</{tag}>' for value in values)
    display(HTML('<table><thead><tr>' + cells(headers, 'th') + '</tr></thead><tbody>' +
                 ''.join('<tr>' + cells(row, 'td') + '</tr>' for row in rows) + '</tbody></table>'))

assert 'topomt' not in sys.modules

## The fixed atomic input

The system contains **94 argon balls of radius 1.88 Å**. The generator starts
from two sampled nominal spheres of radius 7 Å, with centers 12.5 Å apart on
the x axis. It removes the overlapping caps and applies fixed jitter of 0.1 Å
with seed zero. The nominal sampling parameter is 3.5 Å.

Those construction parameters explain the shape; the **actual full-precision
ball coordinates and radii** define the physical obstacle set. We do not assume
that an ideal smooth peanut is sealed. The rounded PDB supplies topology only;
native receiving restores the coordinates from `input.json` and verifies radii.

The two reference points are the nominal chamber centers, x = −6.25 and +6.25 Å.
We will call them the left and right **anchors**. Their behavior makes a precise,
bounded question: are these free points enclosed, and can they communicate?
It does not count every possible region in the system.

In [ ]:
refs = study.references(CASE)
assert 'topomt' not in sys.modules
anchor_clearances = puw.get_value(refs['anchor_clearances'], to_unit='angstroms')
path_clearance = float(puw.get_value(refs['connecting_axis_minimum_clearance'], to_unit='angstroms'))
table(['Probe (Å)', 'Independent statement', 'Scope'], [
    [1.4, 'Both anchors are free, enclosed and connected', 'One common named region; not every global region'],
    [2.2, 'Both anchors remain free, enclosed and separated', 'At least two distinct enclosed regions'],
    [6.0, 'No strict-free center anywhere in the finite native hull', 'The unbounded exterior still exists']])

## Communication needs a complete path

For a probe center x, clearance is the smallest distance to any atomic center
minus that atom's radius. Strict freedom requires clearance greater than the
probe radius. The reference checks each **complete segment** of the axial
polyline against every atom, including closest points between its stations.

The minimum path clearance is **1.432602186 Å**, leaving a **0.032602186 Å**
margin at 1.4 Å. Both anchors have much larger clearances, about 4.97 and 4.89 Å.
The neck therefore controls this particular connecting witness. We have proved
one route, not a globally optimal bottleneck or an exact transition radius.

In [ ]:
geometry = study.read(ARTIFACTS / 'reference_geometry.json')
profile = puw.get_value(study.quantity(geometry['barrier_profile'], 'barrier_profile'), to_unit='angstroms')
x = np.linspace(-6.25, 6.25, 501)
points = np.column_stack([x, np.zeros_like(x), np.zeros_like(x)])
clearance = (np.linalg.norm(points[:, None] - centers[None], axis=2) - radii).min(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout='constrained')
axes[0].scatter(centers[:, 0], centers[:, 1], s=18, color='#567792', alpha=.65, label='Atomic centers')
for sign in (-1, 1):
    axes[0].plot(profile[:, 0], sign * profile[:, 1], color='#3564a3', lw=1.7,
                 label='Covered reference barrier' if sign == 1 else None)
axes[0].plot([-6.25, 6.25], [0, 0], color='#007c57', lw=2, label='Verified connecting axis')
axes[0].scatter(anchor_values[:, 0], anchor_values[:, 1], color='#007c57', marker='*', s=95)
neck_radius = profile[np.argmin(abs(profile[:, 0])), 1]
axes[0].plot([0, 0], [-neck_radius, neck_radius], color='#b04148', ls='--', label='Neck section')
axes[0].set(xlabel='x (Å)', ylabel='y (Å)', title='Atomic input and independent barrier', aspect='equal')
axes[0].legend(fontsize=7, loc='lower center')
axes[1].plot(x, clearance, color='#007c57', label='All-atom axis clearance')
for probe, color in [(1.4, '#2585ad'), (2.2, '#ba7b20'), (6, '#b04148')]:
    axes[1].axhline(probe, color=color, ls='--', label=f'{probe:g} Å probe')
axes[1].set(xlabel='x (Å)', ylabel='Clearance (Å)', title='Room in the lobes, restriction at the neck')
axes[1].legend(fontsize=8); axes[1].grid(alpha=.2)
plt.show()

The left panel is a projection: markers show atomic centers, rather than full
ball radii. The blue outline belongs to the separate reference polyhedron.
The plotted clearance samples illustrate the shape; the continuous segment
calculation certifies the path.

## Enclosure needs a complete wall

A free path between the anchors would not rule out a leak to the exterior.
We therefore construct a **closed, nonconvex polygonal barrier** around them:
32 angular columns and 16 polar intervals per lobe give 994 reference vertices
and 1,984 triangular faces. The frusta join at the neck; every mesh edge belongs
to exactly two faces, and the positive radial profile defines a simple closed
surface containing both anchors and their connecting axis.

Every boundary triangle is independently checked to lie inside the union of
actual expanded atomic balls at 1.4 Å. A leaf certificate names one real atom
whose expanded ball covers all vertices of that leaf simplex. Convexity then
covers its **entire interior**. Explicit binary subdivisions include both
children at every split, so no uncovered gap can be omitted from a certificate.
Reference vertices never become extra obstacle atoms.

Any continuous escape from an anchor must cross that covered wall. Together
with the connecting path, this proves one **enclosed connected region containing
both anchors**. The barrier is a sufficient enclosure witness, not a physical
surface reconstruction or a free-volume boundary.

In [ ]:
rows = []
for name, meaning in [('wall', 'Complete enclosing wall, 1.4 Å'),
                      ('separator', 'Complete neck section, 2.2 Å'),
                      ('finite_hull', 'Whole native finite carrier, 6 Å')]:
    result = refs['covers'][name]
    rows.append([meaning, result['visited_nodes'], result['leaf_count'],
                 f"{float(puw.get_value(result['minimum_coverage_margin'], to_unit='angstroms')):.9f}"])
table(['Certificate', 'Visited tree nodes', 'Covering leaves', 'Minimum cover margin (Å)'], rows)

## Separation needs a complete section

At 2.2 Å both anchors remain free, but the entire polygonal neck section at
x = 0 is covered by expanded balls. Its 32 triangles fill the complete section;
they are not just selected points or an axial obstruction. The outer wall is
still covered because increasing the probe expands every obstacle ball.

A path between the two anchors would have to cross the covered section while
inside the barrier, or cross the covered outer wall to leave it. Both routes
are excluded. The anchors therefore occupy **distinct enclosed regions**.
This proves at least two regions, without claiming exactly two throughout the
whole free set.

The existing path and separator give bounds on neck closure; they do not supply
the optimal passage radius. At one fixed probe, describing two chambers inside
one connected region is also different from observing two disconnected regions
at a larger probe.

## Exclusion needs the whole finite carrier

For 6 Å we use the **atomic-center convex hull H**, which is the native finite
carrier. It differs from the nonconvex reference barrier above. Its 128 hull-face
tetrahedra form a complete fan from an interior point. Every entire tetrahedron
has a sufficient all-atom expanded-ball cover, verified with the same complete
subdivision-tree principle. Thus no strict-free center remains anywhere in H.

This conclusion does not follow merely because the axis is blocked. Nor does
it remove the unbounded exterior of this isolated, nonperiodic system.

In [ ]:
table(['Reference quantity', 'Value'], [
    ['Left anchor clearance (Å)', f'{anchor_clearances[0]:.9f}'],
    ['Right anchor clearance (Å)', f'{anchor_clearances[1]:.9f}'],
    ['Continuous connecting-axis minimum (Å)', f'{path_clearance:.9f}'],
    ['Covered reference wall faces', refs['barrier_faces']],
    ['Complete neck-section triangles', refs['separator_triangles']],
    ['Native-hull tetrahedra covered', refs['finite_hull_tetrahedra']],
    ['Exact global physical component count', refs['exact_global_free_component_count']],
    ['Reference barrier equals native hull', refs['barrier_equals_native_hull']]])
assert 'topomt' not in sys.modules
print('All independent certificates replayed without importing DFND.')

## What to ask of DFND

The [next notebook](dfnd.ipynb) follows these exact anchors into actual native
and public supports. At 1.4 Å it looks for their common parent and recoverable
chamber/throat motifs; at 2.2 Å it checks their separation; at 6 Å it checks
finite-carrier exclusion and the availability of physical hull decomposition.
Additional catalog objects must be interpreted separately.

An increase in component count as a probe grows is **not inherently a defect**:
the free set shrinks, but a connected region can fragment. The older two-versus-five
native-domain observation at 1.4/2.0 Å needs its own support diagnosis under
[#188](https://github.com/uibcdf/topomt/issues/188). These 1.4/2.2 Å references
do not diagnose all five domains at that different intermediate probe.

This known catalog input is a retrospective development case, not a blind
benchmark or a comparison with another engine. Exact global counts/volumes,
physical chamber boundaries, optimal bottlenecks and biological accuracy remain
outside the established reference scope. See the
[evidence guide](artifacts/dumbbell_v1/README.md) for protocols and replay.